# Model Training + Validation - Linear Models + Non-Linear SVM

In [ ]:
""" Created on March 3, 2023 // @author: Sarah Shi & Ruth Tweedy """

This notebook reads in the "test_train_df.csv" generated in the test_train_split.ipynb Notebook, and then trains all of the ML models within the text with the exception of Neural Networks.

In [ ]:
# Parameters to be run in papermill if choosing to run models simultaneously/faster
input_file = "test_train_df.csv"
output_file = "Normal_Dataset_Baseline.csv" #rename as you wish
alkanes = ['C25', 'C27', 'C29', 'C31', 'C33', 'C35'] #you can drop these as needed but will need to renormalize the dataset
run_description = "Full dataset, 180 validation" #rename as you wish
is_run_interactive = True 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.colors as colors


from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

import numpy as np
from sklearn.svm import SVC

from composition_stats# import ilr, clr

The code below sets standard Figure parameters.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
matplotlib.rc('font',**{'family':'Avenir', 'size': 20})
plt.rcParams['pdf.fonttype'] = 42

## Loading in DF

This code loads in the dataframe you are using. This dataframe has been pre-treated in the test_train_split.ipynb file. It includes synthetic and real data, predesignated as test or train.

In [ ]:
min_df = pd.read_csv(input_file).iloc[:, 2:]
min_df["sample_uid"] = np.arange(len(min_df))
min_df

In [ ]:
train_df = min_df[min_df['split'] == 'train']
test_df = min_df[min_df['split'] == 'test']

# Prepping DF for ML

Below, we convert the woody or grassy designation into a number code (0 or 1) to be used in data treatement further down, and create cat_lab which tracks the entries in the min_df.wg column.

In [ ]:
code = pd.Categorical(min_df['wg']).codes
cat_lab = pd.Categorical(min_df['wg'])

Below, you could edit down your min_df dataframe to only contain certain data classes, e.g. no synthetic data.

In [ ]:
#min_df=min_df[(min_df['split'] == 'train') | (min_df['split'] == 'test')]
#min_df.split

Below, we use the pre-ordained classifications in test_train_df to pull out the training and testing data for both the only real data and the synthetic + real datasets.

In [ ]:
targets = ['clr1', 'clr2', 'clr3', 'clr4', 'clr5', 'clr6']

train_data_x_df = min_df[targets][min_df['split'] == 'train']
train_data_y = min_df['wg'][min_df['split'] == 'train']
train_data_x = train_data_x_df.values

train_data_x_res_df = min_df[targets][(min_df['split'] == 'train') | (min_df['split'] == 'synthetic_train')]
train_data_y_res = min_df['wg'][(min_df['split'] == 'train') | (min_df['split'] == 'synthetic_train')]
train_data_x_res = train_data_x_res_df.values

test_data_x_df = min_df[targets][min_df['split'] == 'test']
test_data_y = min_df['wg'][min_df['split'] == 'test']
test_data_x = test_data_x_df

test_data_x_res_df = min_df[targets][(min_df['split'] == 'test') | (min_df['split'] == 'synthetic_test')]
test_data_y_res = min_df['wg'][(min_df['split'] == 'test') | (min_df['split'] == 'synthetic_test')]
test_data_x_res = test_data_x_res_df

## Best Parameter Search

The code below is used for all ML training, iterating through a range of hyperparameters and then defining the best_parameters that should be applied for the dataset training and validation.

In [ ]:
best_param = {}

def grid(model_type, params, X_train, X_test, Y_train, Y_test):
    grid_search = GridSearchCV(estimator=model_type, param_grid=params, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, Y_train)
    
    all_accuracies = {}
    for i, params in enumerate(grid_search.cv_results_['params']):
        acc = grid_search.cv_results_['mean_test_score'][i]
        all_accuracies[str(params)] = acc
        
    grid_predictions = grid_search.predict(X_test)
  
    print(classification_report(Y_test, grid_predictions))
    
    print(" Results from Grid Search " )
    print("\n The best score across ALL searched params:\n",grid_search.best_score_)
    print("\n The best parameters across ALL searched params:\n",grid_search.best_params_)
    print(list(grid_search.best_params_.values()))
    
    global best_param
    best_param = grid_search.best_params_

# Random Forest Classifier

In [ ]:
rf = RandomForestClassifier()

Using the array train_data_x below was giving an error so I am now using it in dataframe form.

In [ ]:
max_depth_range = list(range(1, 16))
accuracy_train = []
accuracy_test = []

for depth in max_depth_range:
    clf = RandomForestClassifier(max_depth = depth, 
        random_state = 42)
    clf.fit(train_data_x_df, train_data_y)
    score_train = clf.score(train_data_x_df, train_data_y)
    accuracy_train.append(score_train)
    score_test = clf.score(test_data_x, test_data_y)
    accuracy_test.append(score_test)

plt.figure(figsize = (8, 8))
plt.plot(list(range(1, 16)), accuracy_train, label = 'Train')
plt.plot(list(range(1, 16)), accuracy_test, label = 'Test')
plt.xlabel('Maximum Depth')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

The code below sets the range of hyperparameters to cycle through.

In [ ]:
param_grid = {
    'n_estimators': [5, 11, 30, 50, 80, 100],
    'max_depth': [2,4],
    'max_features': ['sqrt', 'log2'],#1, 2, 4],
    'min_samples_split': [9, 15, 30, 40],
    'min_samples_leaf': [1, 2, 4, 6],
    'bootstrap': [False, True],
    'criterion': ['gini'],#, 'entropy', 'log_loss'], 
    'random_state': [42],
}

## Finding Best Parameters

This code finds the highest overall accuracy combinations for the Random Forests Code with the SMOTE dataset.

In [ ]:
grid(rf, param_grid, train_data_x_res, test_data_x_res, train_data_y_res, test_data_y_res)

These are the hyperparameter values and combinations that are giving the highest accuracy score overall.

In [ ]:
best_param

## Confusion Matrices

In [ ]:
rf_model = RandomForestClassifier(**best_param, n_jobs=1)
rf_model.fit(train_data_x_res_df, train_data_y_res)

rf_train_res_pred = rf_model.predict(train_data_x_res_df)
rf_train_pred = rf_model.predict(train_data_x_df)
rf_test_res_pred = rf_model.predict(test_data_x_res_df)
rf_test_pred = rf_model.predict(test_data_x_df)

In [ ]:
SEED = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def kfold_eval_classifier(X, y, model, label=""):
    scores = cross_validate(
        model,
        X, y,
        cv=cv,
        scoring=["accuracy", "balanced_accuracy", "f1_macro"],
        n_jobs=1
    )

    print(f"\n[{label}]")
    print("CV accuracy:       ", scores["test_accuracy"].mean(), "±", scores["test_accuracy"].std())
    print("CV balanced acc:  ", scores["test_balanced_accuracy"].mean(), "±", scores["test_balanced_accuracy"].std())
    print("CV F1 macro:      ", scores["test_f1_macro"].mean(), "±", scores["test_f1_macro"].std())

    return scores

In [ ]:
kfold_eval_classifier(train_data_x_df,     train_data_y,     rf_model, label="No SMOTE")
kfold_eval_classifier(train_data_x_res_df, train_data_y_res, rf_model, label="SMOTE")

In [ ]:
mapping = dict(zip(code, cat_lab))
sort_mapping = dict(sorted(mapping.items(), key=lambda item: item[0]))

In [ ]:
rf_cm_train = confusion_matrix(train_data_y_res, rf_train_res_pred)
rf_cm_train_true = confusion_matrix(train_data_y, rf_train_pred)
rf_cm_test = confusion_matrix(test_data_y_res, rf_test_res_pred)
rf_cm_test_true = confusion_matrix(test_data_y, rf_test_pred)

class_labels = list(sort_mapping.values())

In [ ]:
cmap = plt.get_cmap('Blues')

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

new_cmap = truncate_colormap(cmap, 0, 0.6)

In [ ]:
def plot_confusion_matrix(cm, class_labels, title):
    cm = cm.T

    cm_with_sums = np.zeros((cm.shape[0] + 1, cm.shape[1] + 1), dtype=cm.dtype)
    cm_with_sums[:cm.shape[0], :cm.shape[1]] = cm
    cm_with_sums[-1, -1] = np.sum(cm)
    cm_with_sums[:-1, -1] = np.sum(cm, axis=1)
    cm_with_sums[-1, :-1] = np.sum(cm, axis=0)


    plt.rcParams.update({'font.size': 14})

    plt.figure(figsize=(5, 3))
    plt.imshow(cm_with_sums, cmap=new_cmap, interpolation='nearest')
    
 

    class_labels_with_sums = class_labels + ['Total']
    
    plt.xticks(ticks=range(len(class_labels_with_sums)), labels=class_labels_with_sums, rotation=45)
    plt.yticks(ticks=range(len(class_labels_with_sums)), labels=class_labels_with_sums)
    plt.xlabel('Label')
    plt.ylabel('Predicted')
    plt.title(title)

    for i in range(len(class_labels_with_sums)):
        for j in range(len(class_labels_with_sums)):
            if i == len(class_labels_with_sums) - 1 or j == len(class_labels_with_sums) - 1:
                cell_value = cm_with_sums[i, j]
                if i == len(class_labels_with_sums) - 1 and j == len(class_labels_with_sums) - 1:
                    total_diagonal_sum = np.sum(np.diag(cm))
                    if cell_value != 0:
                        percentage = total_diagonal_sum / cell_value
                        plt.text(j, i, f"{int(cell_value)}\n{percentage:.0%}", ha='center', va='center', color='black', fontweight='bold')
                    else:
                        plt.text(j, i, str(int(cell_value)), ha='center', va='center', color='black', fontweight='bold')
                elif i == len(class_labels_with_sums) - 1:
                    pred_sum = cm_with_sums[-1, j]
                    diagonal_val = cm_with_sums[j, j]
                    if pred_sum != 0:
                        percentage = diagonal_val / pred_sum
                        plt.text(j, i, f"{int(cell_value)}\n{percentage:.0%}", ha='center', va='center', color='black')
                    else:
                        plt.text(j, i, str(int(cell_value)), ha='center', va='center', color='black')
                else:
                    true_sum = cm_with_sums[i, -1]
                    pred_sum = cm_with_sums[-1, j]
                    diagonal_val = cm_with_sums[i, i]
                    if true_sum != 0:
                        percentage = diagonal_val / true_sum
                        plt.text(j, i, f"{int(cell_value)}\n{percentage:.0%}", ha='center', va='center', color='black')
                    else:
                        plt.text(j, i, str(int(cell_value)), ha='center', va='center', color='black')
            else:
                plt.text(j, i, str(int(cm_with_sums[i, j])), ha='center', va='center', color='black')

    plt.colorbar()

plt.figure()
plot_confusion_matrix(rf_cm_train, class_labels, 'RF Training')
plt.show()

plt.figure()
plot_confusion_matrix(rf_cm_train_true, class_labels, 'RF Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(rf_cm_test, class_labels, 'RF Validation')

plt.figure()
plot_confusion_matrix(rf_cm_test_true, class_labels, 'RF Validation Excluding Synthetic')

## Visualizing Feature Importance & ROC

In [ ]:
feature_imp = pd.Series(rf_model.feature_importances_,index=alkanes).sort_values(ascending=False)

In [ ]:
for feature, score in feature_imp.sort_values(ascending=False).items():
    print(f"{feature}: {score:.3f}")

In [ ]:
sns.barplot(x=feature_imp, y=feature_imp.index)
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title("Visualizing Important Features RF")
plt.legend()
plt.show()

In [ ]:
reverse_mapping = {v: k for k, v in sort_mapping.items()}
test_data_y_res_code = [reverse_mapping[label] for label in test_data_y_res]
rf_test_res_pred_code = [reverse_mapping[label] for label in rf_test_res_pred]

The code below plots an ROC curve for the RF classifier. A PERFECT model would have an area under the curve of 1. A completely random classifier would have an area under the curve of 0.5.

In [ ]:
rf_predictions = rf_model.predict_proba(test_data_x_res)

In [ ]:
fpr, tpr, thresholds = roc_curve(test_data_y_res_code, rf_predictions[:, 1])

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve for RF')
plt.legend(loc='lower right')
plt.show()

## Incorrect Data

In [ ]:
def incorrect(train_df, train_pred, train_actual, test_df, test_pred, test_actual, min_df):
    incorrect_train_mask = train_pred != train_actual
    incorrect_test_mask = test_pred != test_actual
    
    incorrect_train = train_df[incorrect_train_mask]
    incorrect_test = test_df[incorrect_test_mask]
    
    incorrect = pd.concat([incorrect_train, incorrect_test])
    incorrect = incorrect.assign(Predicted = 'Incorrect')

    correct_mask = ~min_df[(min_df['split'] == 'train') | (min_df['split'] == 'test')].isin(incorrect)

    correct = min_df[(min_df['split'] == 'train') | (min_df['split'] == 'test')][correct_mask.all(axis=1)]
    correct = correct.assign(Predicted = 'Correct')
    
    return incorrect, correct

In [ ]:
rf_incorrect, rf_correct = incorrect(train_df, rf_train_pred, train_data_y, test_df, rf_test_pred, test_data_y, min_df)

In [ ]:
def correct_incorrect_box_plots(model_type, correct, incorrect):

    # Variables to plot
    variables = alkanes

    # Create a figure with four subplots (Grassy Correct, Grassy Incorrect, Woody Correct, Woody Incorrect)
    fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharey=True, sharex=True)
    fig.suptitle(model_type, fontsize=30)

    # Plot box and whisker plots for Grassy Correct dataset
    grassy_correct_data = correct[correct['wg'] == 'Grassy'][variables]
    sns.boxplot(data=grassy_correct_data, ax=axes[0, 0], color='orange',)
    axes[0, 0].set_title(f'Grassy Correct, n={len(grassy_correct_data)}')
    axes[0, 0].set_ylabel('Values')
    #ax.set_title(f'{title} (n={n})')

    # Plot box and whisker plots for Grassy Incorrect dataset
    grassy_incorrect_data = incorrect[incorrect['wg'] == 'Grassy'][variables]
    sns.boxplot(data=grassy_incorrect_data, ax=axes[0, 1], color='red')
    axes[0, 1].set_title(f'Grassy Incorrect, n={len(grassy_incorrect_data)}')

    # Plot box and whisker plots for Woody Correct dataset
    woody_correct_data = correct[correct['wg'] == 'Woody'][variables]
    sns.boxplot(data=woody_correct_data, ax=axes[1, 0], color='lightgreen')
    axes[1, 0].set_title(f'Woody Correct, n={len(woody_correct_data)}')
    axes[1, 0].set_ylabel('Values')
    axes[1, 0].set_xlabel('Variables')

    # Plot box and whisker plots for Woody Incorrect dataset
    woody_incorrect_data = incorrect[incorrect['wg'] == 'Woody'][variables]
    sns.boxplot(data=woody_incorrect_data, ax=axes[1, 1], color='darkgreen')
    axes[1, 1].set_title(f'Woody Incorrect, n={len(woody_incorrect_data)}')
    axes[1, 1].set_xlabel('Variables')

    # Adjust spacing between the plots
    plt.tight_layout()

    # Show the plots
    plt.show()

In [ ]:
correct_incorrect_box_plots("Random Forests", rf_correct, rf_incorrect)

## Decision Tree Classifier

In [ ]:
accuracy_train = []
accuracy_test = []

for depth in max_depth_range:
    clf = DecisionTreeClassifier(max_depth = depth, 
        random_state = 42)
    clf.fit(train_data_x_df, train_data_y)
    score_train = clf.score(train_data_x_df, train_data_y)
    accuracy_train.append(score_train)
    score_test = clf.score(test_data_x_df, test_data_y)
    accuracy_test.append(score_test)

plt.figure(figsize = (8, 8))
plt.plot(list(range(1, 16)), accuracy_train, label = 'Train')
plt.plot(list(range(1, 16)), accuracy_test, label = 'Test')
plt.xlabel('Maximum Depth')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
dt = DecisionTreeClassifier(random_state=42)

In [ ]:
param_grid = {
    'max_depth': [3,5],
    'max_features': [1, 2, 4, 6],
    'min_samples_split': [3, 7, 20],
    'min_samples_leaf': [1, 5, 15],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'class_weight': [None, 'balanced'],
    'random_state': [42]
}

In [ ]:
grid(dt, param_grid, train_data_x_res, test_data_x_res, train_data_y_res, test_data_y_res)

In [ ]:
best_param

In [ ]:
dt_model = DecisionTreeClassifier(**best_param)
dt_model.fit(train_data_x_res_df, train_data_y_res)

dt_train_res_pred = dt_model.predict(train_data_x_res_df)
dt_train_pred = dt_model.predict(train_data_x_df)
dt_test_res_pred = dt_model.predict(test_data_x_res_df)
dt_test_pred = dt_model.predict(test_data_x_df)

In [ ]:
kfold_eval_classifier(train_data_x_df,     train_data_y,     dt_model, label="No SMOTE")
kfold_eval_classifier(train_data_x_res_df, train_data_y_res, dt_model, label="SMOTE")

In [ ]:
dt_cm_train = confusion_matrix(train_data_y_res, dt_train_res_pred)
dt_cm_train_true = confusion_matrix(train_data_y, dt_train_pred)
dt_cm_test = confusion_matrix(test_data_y_res, dt_test_res_pred)
dt_cm_test_true = confusion_matrix(test_data_y, dt_test_pred)

In [ ]:
plt.figure()
plot_confusion_matrix(dt_cm_train, class_labels, 'DT Training')
plt.show()

plt.figure()
plot_confusion_matrix(dt_cm_train_true, class_labels, 'DT Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(dt_cm_test, class_labels, 'DT Validation')

plt.figure()
plot_confusion_matrix(dt_cm_test_true, class_labels, 'DT Validation Excluding Synthetic')

In [ ]:
feature_imp = pd.Series(dt_model.feature_importances_,index=alkanes).sort_values(ascending=False)

In [ ]:
for feature, score in feature_imp.sort_values(ascending=False).items():
    print(f"{feature}: {score:.3f}")

In [ ]:
sns.barplot(x=feature_imp, y=feature_imp.index)
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title("Visualizing Important Features DT")
plt.legend()
plt.show()

In [ ]:
dt_predictions = dt_model.predict_proba(test_data_x_res)

In [ ]:
fpr, tpr, thresholds = roc_curve(test_data_y_res_code, dt_predictions[:, 1])

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve for DT')
plt.legend(loc='lower right')
plt.show()

In [ ]:
dt_incorrect, dt_correct = incorrect(train_df, dt_train_pred, train_data_y, test_df, dt_test_pred, test_data_y, min_df)

In [ ]:
correct_incorrect_box_plots("Decision Tree", dt_correct, dt_incorrect)

## Linear SVM

In [ ]:
svc = SVC()

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'probability' : [True],
    'kernel': ['linear'],
    'class_weight' : ['balanced'],
    'random_state': [42]
}

In [ ]:
grid(svc, param_grid, train_data_x_res_df, test_data_x_res_df, train_data_y_res, test_data_y_res)

In [ ]:
svc_model = SVC(**best_param)
svc_model.fit(train_data_x_res_df, train_data_y_res)

svc_train_res_pred = svc_model.predict(train_data_x_res_df)
svc_train_pred = svc_model.predict(train_data_x_df)
svc_test_res_pred = svc_model.predict(test_data_x_res_df)
svc_test_pred = svc_model.predict(test_data_x_df)

In [ ]:
kfold_eval_classifier(train_data_x_df,     train_data_y,     svc_model, label="No SMOTE")
kfold_eval_classifier(train_data_x_res_df, train_data_y_res, svc_model, label="SMOTE")

In [ ]:
svc_cm_train = confusion_matrix(train_data_y_res, svc_train_res_pred)

svc_cm_train_true = confusion_matrix(train_data_y, svc_train_pred)

svc_cm_test = confusion_matrix(test_data_y_res, svc_test_res_pred)

svc_cm_test_true = confusion_matrix(test_data_y, svc_test_pred)

In [ ]:
plt.figure()
plot_confusion_matrix(svc_cm_train, class_labels, 'Linear SVC Training')
plt.show()

plt.figure()
plot_confusion_matrix(svc_cm_train_true, class_labels, 'Linear SVC Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(svc_cm_test, class_labels, 'Linear SVC Validation')

plt.figure()
plot_confusion_matrix(svc_cm_test_true, class_labels, 'Linear SVC Validation Excluding Synthetic')

In [ ]:
feature_importance = np.abs(svc_model.coef_).mean(axis=0)

feature_imp = pd.Series(feature_importance, index=alkanes).sort_values(ascending=False)

print(feature_imp)

In [ ]:
sns.barplot(x=feature_imp, y=feature_imp.index)
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title("Visualizing Important Features SVC Linear")
plt.legend()
plt.show()

In [ ]:
svc_predictions = svc_model.predict_proba(test_data_x_res)

In [ ]:
fpr, tpr, thresholds = roc_curve(test_data_y_res_code, svc_predictions[:, 1])

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve for Linear SVC')
plt.legend(loc='lower right')
plt.show()

In [ ]:
svc_incorrect, svc_correct = incorrect(train_df, svc_train_pred, train_data_y, test_df, svc_test_pred, test_data_y, min_df)

In [ ]:
correct_incorrect_box_plots("SVC", svc_correct, svc_incorrect)

## SVM Non-Linear

In [ ]:
svc_nl = SVC()

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],#, 100],
    'kernel': ['rbf'],#'linear']#, 'sigmoid'],#'poly'],
    'gamma': [0.001, 0.01, 0.05],#, 10, 100]#,'scale', 'auto'
    'random_state': [42]
}

In [ ]:
grid(svc_nl, param_grid, train_data_x_res_df, test_data_x_res_df, train_data_y_res, test_data_y_res)

In [ ]:
svc_nl_model = SVC(**best_param)
svc_nl_model.fit(train_data_x_res_df, train_data_y_res)

svc_nl_train_res_pred = svc_nl_model.predict(train_data_x_res_df)
svc_nl_train_pred = svc_nl_model.predict(train_data_x_df)
svc_nl_test_res_pred = svc_nl_model.predict(test_data_x_res_df)
svc_nl_test_pred = svc_nl_model.predict(test_data_x_df)

In [ ]:
kfold_eval_classifier(train_data_x_df, train_data_y, svc_nl_model, label="No SMOTE")
kfold_eval_classifier(train_data_x_res_df, train_data_y_res, svc_nl_model, label="SMOTE")

In [ ]:
svc_nl_cm_train = confusion_matrix(train_data_y_res, svc_nl_train_res_pred)

svc_nl_cm_train_true = confusion_matrix(train_data_y, svc_nl_train_pred)

svc_nl_cm_test = confusion_matrix(test_data_y_res, svc_nl_test_res_pred)

svc_nl_cm_test_true = confusion_matrix(test_data_y, svc_nl_test_pred)

In [ ]:
plt.figure()
plot_confusion_matrix(svc_nl_cm_train, class_labels, 'Non-Linear SVC Training')
plt.show()

plt.figure()
plot_confusion_matrix(svc_nl_cm_train_true, class_labels, 'Non-Linear SVC Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(svc_nl_cm_test, class_labels, 'Non-Linear SVC Validation')

plt.figure()
plot_confusion_matrix(svc_nl_cm_test_true, class_labels, 'Non-Linear SVC Validation Excluding Synthetic')



No feature importance here as only works for a linear kernel, same for ROC curves without probability turned on.

In [ ]:
svc_nl_incorrect, svc_nl_correct = incorrect(train_df, svc_nl_train_pred, train_data_y, test_df, svc_nl_test_pred, test_data_y, min_df)

In [ ]:
correct_incorrect_box_plots("SVC NL", svc_nl_correct, svc_nl_incorrect)

## K Neighbors Classifier

In [ ]:
knn= KNeighborsClassifier()

In [ ]:
param_grid = {
    'n_neighbors': [5, 7],
    #'weights': ['distance','uniform'], 
    #'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'leaf_size': [3, 5]
    #'p': [1, 2, 4]
}

In [ ]:
grid(knn, param_grid, train_data_x_res_df, test_data_x_res_df, train_data_y_res, test_data_y_res)

In [ ]:
knn_model = KNeighborsClassifier(**best_param)
knn_model.fit(train_data_x_res_df, train_data_y_res)

knn_train_res_pred = knn_model.predict(train_data_x_res_df)
knn_train_pred = knn_model.predict(train_data_x_df)
knn_test_res_pred = knn_model.predict(test_data_x_res_df)
knn_test_pred = knn_model.predict(test_data_x_df)

In [ ]:
kfold_eval_classifier(train_data_x_df, train_data_y, knn_model, label="No SMOTE")
kfold_eval_classifier(train_data_x_res_df, train_data_y_res, knn_model, label="SMOTE")

In [ ]:
knn_cm_train = confusion_matrix(train_data_y_res, knn_train_res_pred)

knn_cm_train_true = confusion_matrix(train_data_y, knn_train_pred)

knn_cm_test = confusion_matrix(test_data_y_res, knn_test_res_pred)

knn_cm_test_true = confusion_matrix(test_data_y, knn_test_pred)

In [ ]:
plt.figure()
plot_confusion_matrix(knn_cm_train, class_labels, 'KNN Training')
plt.show()

plt.figure()
plot_confusion_matrix(knn_cm_train_true, class_labels, 'KNN Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(knn_cm_test, class_labels, 'KNN Validation')

plt.figure()
plot_confusion_matrix(knn_cm_test_true, class_labels, 'KNN Validation Excluding Synthetic')

In [ ]:
knn_predictions = knn_model.predict_proba(test_data_x_res)

In [ ]:
fpr, tpr, thresholds = roc_curve(test_data_y_res_code, knn_predictions[:, 1])

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC) Curve for KNN')
plt.legend(loc='lower right')
plt.show()

In [ ]:
knn_incorrect, knn_correct = incorrect(train_df, knn_train_pred, train_data_y, test_df, knn_test_pred, test_data_y, min_df)
correct_incorrect_box_plots("KNN", knn_correct, knn_incorrect)

## Gaussian Naive Bayes

In [ ]:
gnb = GaussianNB()

In [ ]:
param_grid = {
    'var_smoothing': np.logspace(0,-9, num=100)}

In [ ]:
grid(gnb, param_grid, train_data_x_res_df, test_data_x_res_df, train_data_y_res, test_data_y_res)

In [ ]:
gnb_model = GaussianNB(**best_param)
gnb_model.fit(train_data_x_res_df, train_data_y_res)

gnb_train_res_pred = gnb_model.predict(train_data_x_res_df)
gnb_train_pred = gnb_model.predict(train_data_x_df)
gnb_test_res_pred = gnb_model.predict(test_data_x_res_df)
gnb_test_pred = gnb_model.predict(test_data_x_df)

In [ ]:
kfold_eval_classifier(train_data_x_df, train_data_y, gnb_model, label="No SMOTE")
kfold_eval_classifier(train_data_x_res_df, train_data_y_res, gnb_model, label="SMOTE")

In [ ]:
gnb_cm_train = confusion_matrix(train_data_y_res, gnb_train_res_pred)

gnb_cm_train_true = confusion_matrix(train_data_y, gnb_train_pred)

gnb_cm_test = confusion_matrix(test_data_y_res, gnb_test_res_pred)

gnb_cm_test_true = confusion_matrix(test_data_y, gnb_test_pred)

In [ ]:
plt.figure()
plot_confusion_matrix(gnb_cm_train, class_labels, 'GNB Training')
plt.show()

plt.figure()
plot_confusion_matrix(gnb_cm_train_true, class_labels, 'GNB Training Excluding Synthetic')
plt.show()

plt.figure()
plot_confusion_matrix(gnb_cm_test, class_labels, 'GNB Validation')

plt.figure()
plot_confusion_matrix(gnb_cm_test_true, class_labels, 'GNB Validation Excluding Synthetic')

In [ ]:
gnb_predictions = gnb_model.predict_proba(test_data_x_res)

In [ ]:
fpr, tpr, thresholds = roc_curve(test_data_y_res_code, gnb_predictions[:, 1])

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC) Curve for GNB')
plt.legend(loc='lower right')
plt.show()

In [ ]:
gnb_incorrect, gnb_correct = incorrect(train_df, gnb_train_pred, train_data_y, test_df, gnb_test_pred, test_data_y, min_df)
correct_incorrect_box_plots("GNB", gnb_correct, gnb_incorrect)

# Total Incorrect

In [ ]:
df_list = [rf_incorrect, dt_incorrect, svc_incorrect, knn_incorrect, gnb_incorrect, svc_nl_incorrect]  # Add all your dataframes here

combined_df = df_list[0].copy()
combined_df['row_count'] = 1

for df in df_list[1:]:
    common_indices = combined_df.index.isin(df.index)
    combined_df.loc[common_indices, 'row_count'] += 1
    
    unique_rows = df.loc[~df.index.isin(combined_df.index)]
    combined_df = pd.concat([combined_df, unique_rows])

combined_df['row_count'].fillna(1, inplace=True)

combined_df = combined_df.sort_values('row_count', ascending=False)
combined_df.reset_index(drop=True, inplace=True)

In [ ]:
combined_df

In [ ]:
key_cols = ['source', 'ID', 'species', 'loc_sure', 'lon', 'lat', 'altitude', 'family', 'subfam', 'dom']

# --- make sure row_count exists in combined_df and is numeric ---
combined_df = combined_df.copy()
combined_df["row_count"] = pd.to_numeric(combined_df.get("row_count", 1), errors="coerce").fillna(1).astype(int)

# --- keep ONLY keys + row_count from combined_df (prevents overwriting other columns) ---
combined_counts = (
    combined_df[key_cols + ["row_count"]]
    .groupby(key_cols, dropna=False, as_index=False)["row_count"]
    .max()
)

# --- merge onto min_df (full set), row_count missing => 0 (i.e., always correct) ---
merged_df = min_df.merge(combined_counts, on=key_cols, how="left")

merged_df["row_count"] = merged_df["row_count"].fillna(0).astype(int)

# Optional: drop real missing / string-"nan" sources robustly
merged_df = merged_df[
    merged_df["source"].notna() &
    (merged_df["source"].astype(str).str.strip().str.lower() != "nan")
]

# Sanity checks
print("min_df rows:", len(min_df))
print("merged_df rows:", len(merged_df))
print(merged_df["row_count"].value_counts().sort_index())

In [ ]:
merged_df = merged_df[merged_df['source'].notna()]
merged_df